# AI014 - Metric Execution Demo

This notebook tests the metric execution code for AI014. It uses existing baseline prediction output when available. If the file is not available, it uses a small fallback sample based on the AI009 validation output.

In [4]:
from pathlib import Path
import sys
import json
import pandas as pd

# Find evaluation root whether the notebook is run from repo root or from the notebooks folder.
cwd = Path.cwd()
if cwd.name == "notebooks":
    evaluation_root = cwd.parent
elif (cwd / "src" / "metrics").exists():
    evaluation_root = cwd
elif (cwd / "ai-ml" / "evaluation").exists():
    evaluation_root = cwd / "ai-ml" / "evaluation"
else:
    evaluation_root = cwd.parent

src_path = evaluation_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from metrics.metric_utils import evaluate_prediction_dataframe, save_metrics_json
from core.io_utils import (
    save_json, load_json,
    save_csv, load_csv,
    save_yaml, load_yaml,
    save_with_timestamp
)
from shared.plotting import plot_confusion_matrix, plot_roc_curve, generate_evaluation_plots

evaluation_root

WindowsPath('d:/BRO/Uni/Group-A/testestest/Phoenix/ai-ml/evaluation')

In [5]:
# Try to load AI009 baseline predictions. If not found, use a small fallback sample.
candidate_paths = [
    evaluation_root.parent / "ai009_baseline" / "outputs" / "predictions_val.csv",
    evaluation_root / "predictions_val.csv",
    Path.cwd() / "predictions_val.csv",
]

input_path = next((path for path in candidate_paths if path.exists()), None)

if input_path:
    df = pd.read_csv(input_path)
    source = str(input_path)
else:
    df = pd.DataFrame({
        "temperature": [29, 47],
        "humidity": [62, 27],
        "actual_label": [0, 1],
        "predicted_label": [0, 1],
        "predicted_probability": [0.23794634949659663, 0.9982779224071671],
    })
    source = "fallback sample based on AI009 validation output"

print(f"Input source: {source}")
df

Input source: fallback sample based on AI009 validation output


,temperature,humidity,actual_label,predicted_label,predicted_probability
0,29,62,0,0,0.237946
1,47,27,1,1,0.998278


In [3]:
result = evaluate_prediction_dataframe(
    df,
    actual_col="actual_label",
    predicted_col="predicted_label",
    probability_col="predicted_probability",
    metadata={"source": source},
)

print(json.dumps(result["metrics"], indent=2))

{
  "accuracy": 1.0,
  "precision": 1.0,
  "recall": 1.0,
  "f1_score": 1.0,
  "false_positive_rate": 0.0,
  "false_negative_rate": 0.0,
  "roc_auc": 1.0,
  "confusion_matrix": [
    [
      1,
      0
    ],
    [
      0,
      1
    ]
  ],
  "counts": {
    "tp": 1,
    "tn": 1,
    "fp": 0,
    "fn": 0
  },
  "rows_evaluated": 2
}


In [6]:
output_path = evaluation_root / "outputs" / "metrics" 
save_metrics_json(result, output_path / "evaluation_metrics.json")
print(f"Saved metric output to: {output_path}")

# JSON
save_json(result, output_path / "test.json")
loaded_json = load_json(output_path / "test.json")
print("JSON OK:", loaded_json == result)

# CSV 
save_csv(df, output_path / "test.csv")
loaded_csv = load_csv(output_path / "test.csv")
print("CSV OK:", not loaded_csv.empty)

# YAML
save_yaml(result, output_path / "test.yaml")
loaded_yaml = load_yaml(output_path / "test.yaml")
print("YAML OK:", loaded_yaml == result)

# Timestamp Save
path = save_with_timestamp(result, output_path, "timestamp_test")
print("Timestamp file created:", path)


Saved metric output to: d:\BRO\Uni\Group-A\testestest\Phoenix\ai-ml\evaluation\outputs\metrics
JSON OK: True
CSV OK: True
YAML OK: True
Timestamp file created: d:\BRO\Uni\Group-A\testestest\Phoenix\ai-ml\evaluation\outputs\metrics\timestamp_test_20260514_083043.json


In [12]:
plot_path = evaluation_root / "outputs" / "plots"

# Sample data
y_true = [0, 1, 1, 0, 1, 0, 1, 1, 0, 1]
y_pred = [0, 1, 0, 0, 1, 1, 0, 1, 0, 1]
y_prob = [0.1, 0.9, 0.4, 0.2, 0.8, 0.7, 0.3, 0.8, 0.3, 0.6]

# Confusion matrix (TN, FP, FN, TP → [[TN, FP], [FN, TP]])
cm = [[4, 1], [2, 3]]

# Test individual functions 

plot_confusion_matrix(
    cm,
    labels=["Normal", "Anomaly"],
    save_path= plot_path / "test_confusion_matrix.png"
)

plot_roc_curve(
    y_true,
    y_prob,
    save_path= plot_path / "test_roc_curve.png"
)

# Test full pipeline helper
results = {
    "confusion_matrix": cm
}

generate_evaluation_plots(
    results=results,
    y_true=y_true,
    y_prob=y_prob,
    model_name="TestModel",
    dataset_name="TestDataset"
)

print("Plotting test completed. Check outputs/plots/")

Plotting test completed. Check outputs/plots/
